# Malilangwe Wildlife Detector — WAID Fine-Tuning

Fine-tunes YOLOv11 on the WAID aerial wildlife dataset.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Edit the Settings cell below if needed
3. `Runtime → Run all`

**Hitting rate limits?** Use `EPOCHS=30`, `BATCH=8` (~50 min). Download `last.pt` at the end of each session and re-upload next time to resume.

## ⚙️ Settings — Edit These Before Running

In [ ]:
# How many epochs to train this session
# First time: 30 (~50 min, safe for free Colab)
# Full run:   100 (~2-3 hrs, best results)
EPOCHS = 30

# Batch size — 8 is safe for free T4, 16 is faster but may crash
BATCH = 8

# Resuming from a previous session?
# False = start fresh | True = upload last.pt when prompted
RESUME = False

print(f'Settings: EPOCHS={EPOCHS}, BATCH={BATCH}, RESUME={RESUME}')

## 1. Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU, then re-run.')
print(r.stdout)

## 2. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/Tadiwa-M/wildlife-detector-malilangwe.git'
REPO_DIR = '/content/wildlife-detector-malilangwe'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 3. Upload Previous Checkpoint (Resume only)

**Skip if `RESUME = False`.** If resuming, upload your `last.pt` when prompted.

In [ ]:
CHECKPOINT_PATH = None

if RESUME:
    from google.colab import files as colab_files
    from pathlib import Path
    print('Upload your last.pt checkpoint...')
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded. Upload last.pt or set RESUME=False.')
    name = list(uploaded.keys())[0]
    dest = Path('runs/train/waid_yolo11n/weights')
    dest.mkdir(parents=True, exist_ok=True)
    os.rename(name, dest / 'last.pt')
    CHECKPOINT_PATH = str(dest / 'last.pt')
    print(f'Checkpoint ready: {CHECKPOINT_PATH}')
else:
    print('RESUME=False — starting fresh.')

## 4. Download WAID Images

In [ ]:
import os
WAID_DIR = '/content/wildlife-detector-malilangwe/WAID'

if not os.path.exists(os.path.join(WAID_DIR, 'WAID', 'images')):
    print('Downloading WAID dataset (~2 GB)...')
    !git clone --depth=1 https://github.com/xiaohuicui/WAID.git {WAID_DIR}/WAID_repo
    !cp -r {WAID_DIR}/WAID_repo/WAID/images {WAID_DIR}/WAID/images
    print('Done.')
else:
    print('WAID images already present.')

## 5. Validate Dataset & Generate YAML

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data.dataset import validate_dataset, get_class_distribution, generate_dataset_yaml
from src.utils.logging_setup import setup_logging

cfg = load_config()
setup_logging(cfg)

stats = validate_dataset(cfg)
print(f"\nTotal images : {stats['total_images']:,}")
print(f"Total labels : {stats['total_labels']:,}")

print('\nClass distribution (train):')
dist = get_class_distribution(cfg, split='train')
for name, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {name:<12s} {count:>8,}')

dataset_yaml = generate_dataset_yaml(cfg)
print('\nDataset YAML:', dataset_yaml)

## 6. Train

Watch mAP50 climb each epoch. Early stopping kicks in after 15 epochs with no improvement.

In [ ]:
from ultralytics import YOLO

train_cfg = cfg.training
det_cfg = cfg.detection

if RESUME and CHECKPOINT_PATH:
    print(f'Resuming from {CHECKPOINT_PATH}')
    model = YOLO(CHECKPOINT_PATH)
    resume_flag = True
else:
    print(f'Starting fresh from pretrained {det_cfg.model_variant}.pt')
    model = YOLO(f'{det_cfg.model_variant}.pt')
    resume_flag = False

results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=int(train_cfg.image_size),
    optimizer=str(train_cfg.optimizer),
    lr0=float(train_cfg.learning_rate),
    weight_decay=float(train_cfg.weight_decay),
    patience=int(train_cfg.patience),
    device=0,
    hsv_h=float(train_cfg.augmentation.hsv_h),
    hsv_s=float(train_cfg.augmentation.hsv_s),
    hsv_v=float(train_cfg.augmentation.hsv_v),
    flipud=float(train_cfg.augmentation.flipud),
    fliplr=float(train_cfg.augmentation.fliplr),
    mosaic=float(train_cfg.augmentation.mosaic),
    mixup=float(train_cfg.augmentation.mixup),
    scale=float(train_cfg.augmentation.scale),
    project='runs/train',
    name='waid_yolo11n',
    exist_ok=True,
    resume=resume_flag,
)

print('\nTraining complete!')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

## 7. Evaluate

In [ ]:
from pathlib import Path

best = Path('runs/train/waid_yolo11n/weights/best.pt')
if best.exists():
    eval_model = YOLO(str(best))
    metrics = eval_model.val(
        data=str(dataset_yaml), split='test',
        conf=float(det_cfg.confidence_threshold),
        iou=float(det_cfg.iou_threshold),
        imgsz=int(det_cfg.image_size),
        device=0, plots=True,
    )
    print(f'\nmAP@50:    {metrics.box.map50:.4f}')
    print(f'mAP@50-95: {metrics.box.map:.4f}')
    print(f'Precision:  {metrics.box.mp:.4f}')
    print(f'Recall:     {metrics.box.mr:.4f}')
else:
    print('best.pt not found — did training finish?')

## 8. Download Weights

- `waid_best.pt` — best checkpoint overall (use this locally when done)
- `waid_last.pt` — most recent epoch (re-upload this to resume next session)

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

w = Path('runs/train/waid_yolo11n/weights')
for fname, dest in [('best.pt', '/content/waid_best.pt'), ('last.pt', '/content/waid_last.pt')]:
    src = w / fname
    if src.exists():
        shutil.copy(src, dest)
        files.download(dest)
        print(f'Downloading {fname}...')

print('\nDone!')
print('  Resume next session: set RESUME=True, re-upload waid_last.pt')
print('  Finished training:   put waid_best.pt in your local weights/ folder')

## 9. Test on a Custom Image (Optional)

In [ ]:
from google.colab import files as colab_files
from IPython.display import Image, display
import glob

print('Upload an aerial image...')
uploaded = colab_files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]
    m = YOLO('runs/train/waid_yolo11n/weights/best.pt')
    res = m.predict(source=img_path, conf=0.15, save=True,
                    project='/content/test_output', name='result', exist_ok=True)
    for r in res:
        print(f'Detections: {len(r.boxes)}')
        for box in r.boxes:
            print(f'  {r.names[int(box.cls[0])]}: {float(box.conf[0]):.0%}')
    saved = glob.glob('/content/test_output/result/*.jpg') + glob.glob('/content/test_output/result/*.png')
    if saved:
        display(Image(saved[0]))